In [1]:
from dotenv import load_dotenv
from langgraph.graph import MessagesState
from langgraph.graph import START, StateGraph
from langgraph.prebuilt import tools_condition, ToolNode
from langchain.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from pydantic import BaseModel
from pprint import pprint
from langchain.tools import tool, ToolRuntime
from langgraph.checkpoint.memory import InMemorySaver 
from langgraph.types import interrupt, Command


load_dotenv()

True

In [2]:
from langchain_groq import ChatGroq

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [4]:
import pandas as pd
import json
import pickle
import jupyter_client
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [5]:
class JupyterSandbox:
    def __init__(self):
        self.km, self.kc = jupyter_client.manager.start_new_kernel(kernel_name='python3')
        
    def execute_code(self, code: str) -> str:
        msg_id = self.kc.execute(code)
        reply = self.kc.get_shell_msg(timeout=60)
        output = []
        
        while True:
            try:
                io_msg = self.kc.get_iopub_msg(timeout=0.5)
                msg_type = io_msg['header']['msg_type']
                content = io_msg['content']
                
                if msg_type == 'stream':
                    output.append(content['text'])
                elif msg_type == 'error':
                    output.append(f"Error: {content['ename']}: {content['evalue']}\n" + "".join(content['traceback']))
                elif msg_type == 'execute_result' or msg_type == 'display_data':
                    if 'text/plain' in content['data']:
                        output.append(content['data']['text/plain'])
            except:
                break
                
        full_output = "\n".join(output)
    
        # Define a safe character limit (e.g., ~10,000 characters or ~2,500 tokens)
        MAX_CHAR_LIMIT = 1000
    
        if len(full_output) > MAX_CHAR_LIMIT:
            truncated_output = (
                full_output[:MAX_CHAR_LIMIT] 
                + f"\n\n... [TRUNCATED {len(full_output) - MAX_CHAR_LIMIT} CHARACTERS DUE TO LLM CONTEXT LIMITS] ..."
            )
            return truncated_output
        
        return full_output if full_output else "Code executed successfully."

    def shutdown(self):
        self.km.shutdown_kernel()

In [6]:
sandbox = JupyterSandbox()

In [7]:
@tool
def run_python_code(code: str) -> str:
    """Executes Python code in an isolated local Jupyter kernel sandbox and returns the stdout or errors. 
    Use this to inspect data, clean datasets, train models, and print evaluation metrics."""
    return sandbox.execute_code(code)

In [8]:
print(run_python_code.invoke({
    "code": "print('hello')"
}))

hello



In [9]:
def load_and_profile_data(dataset_path: str) -> str:
    """
    Loads a dataset and generates a comprehensive profile for EDA and preprocessing.

    Creates:
    - Dataset overview (rows, columns, column names)
    - Data types for all columns
    - Missing value counts and percentages
    - Unique value counts for each column
    - Duplicate row count
    - Lists of numerical, categorical, and datetime columns
    - Constant column detection
    - Numerical summary statistics using df.describe()
    - Categorical summary statistics using df.describe()
    - Correlation matrix for numerical features
    - Sample rows for data inspection

    Returns:
    - JSON string with all the necessary information of the dataframe

    """

    df = pd.read_csv(dataset_path)

    numeric_cols = df.select_dtypes(include="number").columns
    categorical_cols = df.select_dtypes(
        include=["object", "category", "bool"]
    ).columns
    datetime_cols = df.select_dtypes(
        include=["datetime64[ns]", "datetime64"]
    ).columns

    profile = {
        # Dataset overview
        "rows": len(df),
        "num_columns": len(df.columns),
        "columns": df.columns.tolist(),

        # Schema
        "dtypes": {
            col: str(dtype)
            for col, dtype in df.dtypes.items()
        },

        # Missing values
        "null_counts": df.isnull().sum().to_dict(),
        "null_percentages": (
            df.isnull().mean() * 100
        ).round(2).to_dict(),

        # Cardinality
        "unique_values": df.nunique().to_dict(),

        # Duplicates
        "duplicate_rows": int(df.duplicated().sum()),

        # Column groups
        "numeric_columns": list(numeric_cols),
        "categorical_columns": list(categorical_cols),
        "datetime_columns": list(datetime_cols),

        # Constant columns
        "constant_columns": [
            col for col in df.columns
            if df[col].nunique(dropna=False) <= 1
        ],

        # Numeric summary statistics
        "numeric_summary": (
            df.describe()
            .round(4)
            .to_dict()
            if len(numeric_cols) > 0
            else {}
        ),

        # Categorical summary statistics
        "categorical_summary": (
            df[categorical_cols]
            .describe()
            .to_dict()
            if len(categorical_cols) > 0
            else {}
        ),

        # Correlation matrix
        "correlation_matrix": (
            df[numeric_cols]
            .corr()
            .round(4)
            .to_dict()
            if len(numeric_cols) > 1
            else {}
        ),

        # Sample rows for context
        "sample_rows": (
            df.head(5)
            .fillna("NULL")
            .to_dict(orient="records")
        ),

        # Percentage of missing values per column
        "null_percentage": (
            (df.isnull().sum() / len(df)) * 100
        ).round(2).to_dict(),

        # Skewness of numerical columns
        "skewness": (
            df.select_dtypes(include="number")
            .skew()
            .round(4)
            .to_dict()
        )
    }

    return json.dumps(profile, default=str)

In [10]:
eda = load_and_profile_data('datasets/ai_student.csv')

In [11]:
from typing import TypedDict, List, Dict, Any, Literal

class InteractiveDataScienceState(MessagesState):
    # Data Paths
    dataset_path: str
    cleaned_data_path: str
    
    # Flow Control Flags
    user_input: str
    current_stage: Literal["eda", "cleaning", "ml_debate", "complete"]
    user_approved_transition: bool
    
    # Internal agent history
    eda_output_to_user: str
    cleaning_plan: str
    model_metrics: Dict[str, float]
    critic_feedback: str
    final_report: str

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0
)
llm_with_tools = llm.bind_tools([run_python_code])

In [13]:
llm_planner = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0
)

In [14]:
def cleaning_plan_node(state: InteractiveDataScienceState) -> Dict[str, Any]:
    print("\n[Node: Cleaning Agent] Processing your dataset...")

    
    sys_msg = SystemMessage(content=(f"""You are a Data Cleaning Expert.

You have been provided with Exploratory Data Analysis (EDA) results for a dataset.

Your task is to carefully analyze the EDA findings and create a comprehensive data cleaning plan before any modeling is performed.

EDA Results:
{state['eda_output_to_user']}

Analyze the EDA results and identify:

1. Missing value issues
   - Columns with missing values
   - Severity of missingness
   - Recommended treatment (drop, mean/median/mode imputation, forward fill, backward fill, interpolation, etc.)
   - Justification for each recommendation

2. Duplicate records
   - Presence of duplicate rows
   - Recommended action

3. Data type issues
   - Columns with potentially incorrect data types
   - Recommended conversions

4. Outliers
   - Numerical columns containing significant outliers
   - Whether to keep, cap, transform, or remove them
   - Justification based on the EDA findings

5. Categorical data issues
   - High-cardinality columns
   - Rare categories
   - Inconsistent labels, casing, spelling, or formatting problems

6. Feature quality issues
   - Constant or near-constant columns
   - Columns with extremely high missingness
   - Columns with little predictive value
   - Potential data leakage columns

7. Distribution-related issues
   - Highly skewed numerical features
   - Recommended transformations if needed

8. Correlation and redundancy
   - Highly correlated features
   - Redundant columns that may require removal

9. Data consistency issues
   - Invalid values
   - Impossible values
   - Formatting inconsistencies
   - Potential anomalies

10. Cleaning execution order
    - Provide the recommended sequence of cleaning steps

Return your response as a structured cleaning plan.

Only make recommendations that are supported by the provided EDA results. Do not invent issues that are not evident from the analysis.
  
    """))



    response = llm_planner.invoke([sys_msg] + state['messages'])
    
    return {
        "cleaning_plan": response.content,
        "messages": [response],
        "current_stage": "cleaning" 
    }

In [15]:
def cleaning_code_executor(state: InteractiveDataScienceState) -> Dict[str, Any]:

    """Pause and ask the user to approve, revise, or no cleaning."""
    if not state.get("user_input"):
      user_review = interrupt({
        "question": "Here's the proposed cleaning plan. Approve, suggest changes, or no cleaning?",
        "proposed_steps": state["cleaning_plan"],
    })
    else:
      user_review = state["user_input"]
    
    
    sys_msg = SystemMessage(content=(f"""You are a Senior Data Cleaning Execution Agent.

Your job is to review the proposed cleaning plan, consider the user's feedback, generate Python code to perform the required cleaning, and execute that code using the available `run_python_code` tool.

Available Information:

Dataset Path:
{state['dataset_path']}

Cleaning Plan:
{state['cleaning_plan']}

User Review / Feedback:
{user_review}

Instructions:

1. Read the cleaning plan carefully.
2. Read the user's feedback carefully.
3. Determine how to proceed:

   A. If the user APPROVED the cleaning plan:
      - Execute the cleaning plan as specified.

   B. If the user requested MODIFICATIONS:
      - Follow the user's instructions.
      - Override the corresponding cleaning methods from the original plan.
      - Keep all other approved cleaning steps unchanged.

   C. If the user explicitly requested NOT to clean the dataset:
      - Do not execute any cleaning code.
      - Return a summary explaining that no cleaning was performed.

4. The dataset is located at:
   state["dataset_path"]

5. Generate Python code that:
   - Loads the dataset.
   - Applies the approved cleaning operations.
   - Saves the cleaned dataset to a new file as 'datasets/cleaned.csv'.
   - Produces useful execution output describing what was changed.

6. Execute the generated code using the `run_python_code` tool.

7. Before executing code:
   - Verify that the code aligns with the approved cleaning plan and user feedback.
   - Ensure all referenced columns exist before applying transformations.
   - Handle potential errors gracefully.

8. Only perform safe data-cleaning operations such as:
   - Missing value treatment
   - Duplicate removal
   - Datatype conversions
   - Outlier treatment
   - Category standardization
   - Feature removal
   - Basic feature transformations

9. Strict Safety Requirements:

   Never generate or execute code that:
   - Uses `eval()`
   - Uses `exec()`
   - Uses `compile()`
   - Uses `subprocess`
   - Uses `os.system`
   - Uses `shutil.rmtree`
   - Deletes files
   - Modifies files outside the dataset workflow
   - Accesses the network
   - Installs packages
   - Executes shell commands
   - Reads arbitrary system files
   - Performs actions unrelated to dataset cleaning

10. Save the cleaned dataset as a new file.
    Never overwrite the original dataset.

11. If code execution fails:
    - Analyze the error.
    - Generate a corrected version of the code.
    - Retry using the `run_python_code` tool.
    - Repeat until the task succeeds or a safe resolution is reached.

AFTER RUNNING THE run_python_code TOOL AND GETTING 'code executed successfully' TOOL OUTPUT:
   - Confirm whether cleaning succeeded.
   - Summarize the performed actions.
   - Report the location of the cleaned dataset.

Output Format:

  "cleaning_performed": true/false,
  "user_changes_applied": [...],
  "cleaning_actions": [...],
  "cleaned_dataset_path": "...",
  "execution_status": "success | failed",
  "summary": "Short summary of what was done."

Always use the `run_python_code` tool to perform the actual cleaning work. Do not merely describe the cleaning steps. The final response should be based on the actual execution results produced by the tool.
  
    """))


    response = llm_with_tools.invoke([sys_msg]+state['messages'])
    
    return {
        "messages": [response],
        "current_stage": "cleaning",
        "user_input": "user_review"
    }

In [16]:
g = StateGraph(InteractiveDataScienceState)
g.add_node('clean_plan_llm',cleaning_plan_node)
g.add_node('clean_code_llm',cleaning_code_executor)
g.add_node('tools',ToolNode([run_python_code]))
g.add_edge(START,'clean_plan_llm')
g.add_edge('clean_plan_llm','clean_code_llm')
g.add_conditional_edges("clean_code_llm", tools_condition)
g.add_edge("tools", "clean_code_llm")
memory = InMemorySaver()
graph = g.compile(checkpointer=memory)

In [ ]:
initial_input = {"messages": HumanMessage(content="Clean the dataset"),
                 'dataset_path':'datasets/ai_student.csv',
                 'eda_output_to_user':eda}

config = {"configurable": {"thread_id": "2"}}
for event in graph.stream(
    initial_input,
    config=config,
    stream_mode="values"
):
    if "__interrupt__" in event:
        interrupt_value = event["__interrupt__"][0].value

        print("\nGraph paused:")
        print(interrupt_value['question'])
        print(interrupt_value['proposed_steps'][0]['text'])


        user_input = input("\nYour response: ")

        for resumed_event in graph.stream(
            Command(resume=user_input),
            config=config,
            stream_mode="values"
        ):
            if "messages" in resumed_event:
                resumed_event["messages"][-1].pretty_print()

    elif "messages" in event:
        event["messages"][-1].pretty_print()

================================ Human Message =================================

Clean the dataset

[Node: Cleaning Agent] Processing your dataset...
================================== Ai Message ==================================

[{'type': 'text', 'text': 'Based on the Exploratory Data Analysis (EDA) provided, here is the comprehensive data cleaning plan.\n\n### 1. Missing Value Issues\n*   **Findings:** There are **no missing values** in the dataset (0% nulls across all 16 columns).\n*   **Recommendation:** No imputation or dropping required.\n\n### 2. Duplicate Records\n*   **Findings:** There are **0 duplicate rows** in the dataset.\n*   **Recommendation:** No action required.\n\n### 3. Data Type Issues\n*   **Findings:** \n    *   `Year_of_Study` is currently an `object`. Since it represents an ordinal progression (Freshman, Sophomore, etc.), it should be treated as an ordinal categorical variable.\n    *   `Paid_Subscription` is `bool`, which is appropriate.\n*   **Recommendati

In [1]:
sandbox.shutdown()

NameError: name 'sandbox' is not defined

In [ ]:
state = graph.get_state(config)

for msg in state.values["messages"]:
    print(type(msg).__name__)

HumanMessage
AIMessage
AIMessage
ToolMessage
AIMessage
